In [31]:
from core.settings import get_settings
from services.multilingual_manager import MultilingualManager
from services.encoder_factory import EncoderFactory
from services.calibrator_factory import CalibratorFactory
from services.model_registry import ModelRegistry
from pyi18next.backends.fs import Backend
from pyi18next.i18next import I18next
from utils.graph_builder import LocalizationGraphBuilder, traverse_namespaces
from adaptation.misc import NameAnonymizer
import os


In [32]:
settings = get_settings()
languages = settings.languages
print(settings)


languages={'es'} spacy={'es': 'es_core_news_sm'} sbert={'es': 'hiiamsid/sentence_similarity_spanish_es'} word2vec={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/vectors.bin'} siamese_lstm={'es': 'C:/Users/malos/Documents/GitHub/JustShare/server/models/bilstm_mean_cosine'} models=[<ModelType.SBERT: 'sbert'>, <ModelType.SPACY: 'spacy'>] allow_origins=['http://localhost:8080', 'http://127.0.0.1:8080'] host='0.0.0.0' port=8000 faiss_data_dir='./faiss_data' adaptation_data_dir='./adaptation/data' localization_dir='./adaptation/localization'


In [33]:
adaptation_dir = "./adaptation"

localization_dir = os.path.join(adaptation_dir, "localization")
language_dir = os.path.join(localization_dir, "dialogue", "active")
structure_dir = os.path.join(localization_dir,  "structure", "modified")

database_dir = "./faiss_data"

data_dir = os.path.join(adaptation_dir, "data")
name_whitelist_path = os.path.join(data_dir, "name_whitelist.txt")
spanish_names_path = os.path.join(data_dir, "nombres-propios-es.txt")


In [34]:
namespaces = traverse_namespaces(language_dir, languages)
print(namespaces)


['scene6/routeA/scene6EndingRouteA', 'generalDialogs', 'scene2/scene2Break', 'scene7/scene7Bedroom', 'menus/titleScene', 'scene4/scene4Bedroom', 'scene6/scene6Livingroom', 'scene4/scene4Frontyard', 'computer/socialMediaScreen', 'dialogManager', 'menus/loginScene', 'scene6/routeB/scene6LunchRouteB', 'scene6/routeB/scene6EndingRouteB', 'scene1/scene1Lunch2', 'scene1/scene1Classroom', 'scene6/routeB/scene6BedroomRouteB', 'scene6/routeA/scene6LunchRouteA', 'scene2/scene2Bedroom', 'scene6/scene6Bedroom', 'computer/captions', 'menus/creditsScene', 'computer/usernames', 'names', 'scene5/scene5Livingroom', 'scene1/scene1Bedroom2', 'scene3/scene3Bedroom', 'scene1/scene1Lunch1', 'scene6/routeA/scene6PortalRouteA', 'transitions', 'computer/loginScreen', 'scene6/routeA/scene6BedroomRouteA1', 'scene6/routeB/scene6PoliceStationRouteB', 'scene1/scene1Break', 'deviceInfo', 'scene1/scene1Bedroom1', 'scene4/scene4Garage', 'scene4/scene4Backyard', 'scene3/scene3Break', 'scene6/routeA/scene6BedroomRouteA2

In [35]:
backend = Backend(
    name_mapping=lambda lng, ns: os.path.join(
        language_dir,
        lng,
        f"{ns}.json"
    )
)

i18n = I18next(
	backend=backend,
	lng=list(languages),
	ns=namespaces,
)



In [36]:
name_anonymizer = NameAnonymizer(
    names_path=spanish_names_path,
    whitelist_path=name_whitelist_path,
    replacement="[UNK]"
)


In [37]:
model_registry = ModelRegistry(languages)
model_registry.build_transformer("sbert")
model_registry.build_lstm()
model_registry.resolve_all()
encoder_factory = EncoderFactory(model_registry)
calibrator_factory = CalibratorFactory(model_registry)
multilingual = MultilingualManager(
    encoder_factory=encoder_factory,
    calibrator_factory=calibrator_factory, 
    name_anonymizer=name_anonymizer,
    base_dir=database_dir
)
model_types = model_registry.active_model_types()


2026-07-06 05:15:32.833 | DEBUG    | services.model_registry:_create_loader:57 - Registering sbert loader for 'es'.
2026-07-06 05:15:32.833 | DEBUG    | services.model_registry:_create_loader:57 - Registering siamese_lstm loader for 'es'.
2026-07-06 05:15:32.833 | DEBUG    | services.model_registry:_create_loader:57 - Registering lstm calibrator loader for 'es'.
2026-07-06 05:15:32.833 | DEBUG    | services.lazy_loader:model:16 - Loading sbert for 'es'...


Using device: cuda


2026-07-06 05:15:34.867 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded sbert for 'es'
2026-07-06 05:15:34.867 | DEBUG    | services.lazy_loader:model:16 - Loading siamese_lstm for 'es'...


Using device: cuda


2026-07-06 05:15:36.154 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded siamese_lstm for 'es'
2026-07-06 05:15:36.154 | DEBUG    | services.lazy_loader:model:16 - Loading lstm calibrator for 'es'...
2026-07-06 05:15:36.154 | SUCCESS  | services.lazy_loader:model:22 - Successfully loaded lstm calibrator for 'es'


In [38]:
builder = LocalizationGraphBuilder(
    i18n=i18n,
    languages=languages,
    multilingual=multilingual,
    model_registry=model_registry,
    base_dir=structure_dir,
)

builder.run()


2026-07-06 05:15:36.391 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 33 vectors
2026-07-06 05:15:36.666 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 33 vectors
2026-07-06 05:15:36.743 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 40 vectors
2026-07-06 05:15:36.940 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 40 vectors
2026-07-06 05:15:37.021 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 31 vectors
2026-07-06 05:15:37.243 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 31 vectors
2026-07-06 05:15:37.326 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 48 vectors
2026-07-06 05:15:37.494 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 48 vectors
2026-07-06 05:15:37.566 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 40 vectors
2026-07-06 05:15:37.733 | DEBUG    | controllers.retrievers.faiss:_fit:95 - Indexed 40 vectors
2026-07-06 05:15:37.828 | DEBUG    | controllers.r

Total visited nodes: 732


In [39]:
multilingual = MultilingualManager(
    encoder_factory=encoder_factory,
    calibrator_factory=calibrator_factory, 
    name_anonymizer=name_anonymizer,
    base_dir=database_dir
)
test_engine = multilingual.get_node_engine("es", "sbert")

print(test_engine.retrievers)

test_engine.load_all()

print(test_engine.retrievers)


2026-07-06 05:15:39.838 | DEBUG    | services.node_engine:load_node:56 - Loading FAISS node | model=sbert | language=es | node=scene1Bedroom1_computer1_choices_similarity
2026-07-06 05:15:39.838 | SUCCESS  | services.node_engine:load_node:71 - Loaded node successfully.
2026-07-06 05:15:39.838 | DEBUG    | services.node_engine:load_node:56 - Loading FAISS node | model=sbert | language=es | node=scene1Bedroom1_computer2_root
2026-07-06 05:15:39.847 | SUCCESS  | services.node_engine:load_node:71 - Loaded node successfully.
2026-07-06 05:15:39.847 | DEBUG    | services.node_engine:load_node:56 - Loading FAISS node | model=sbert | language=es | node=scene1Bedroom2_computer_choices2_similarity
2026-07-06 05:15:39.847 | SUCCESS  | services.node_engine:load_node:71 - Loaded node successfully.
2026-07-06 05:15:39.847 | DEBUG    | services.node_engine:load_node:56 - Loading FAISS node | model=sbert | language=es | node=scene1Classroom_part2_thanks_similarity
2026-07-06 05:15:39.847 | SUCCESS  | 

{}
{'scene1Bedroom1_computer1_choices_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x0000021253A7D010>, 'scene1Bedroom1_computer2_root': <controllers.retrievers.faiss.FaissRetriever object at 0x0000021328F6D640>, 'scene1Bedroom2_computer_choices2_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x000002122C7536E0>, 'scene1Classroom_part2_thanks_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x0000021328F5F050>, 'scene2Break_part2_choice_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x00000213015991C0>, 'scene3Bedroom_main_choices_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x00000212EEA27800>, 'scene4Backyard_mainConversation_choices_similarity': <controllers.retrievers.faiss.FaissRetriever object at 0x00000212EEA24DA0>, 'scene4Bedroom_phone_root': <controllers.retrievers.faiss.FaissRetriever object at 0x00000212EEA24BC0>, 'scene4Garage_phone1_root': <controllers.retrievers.fais

In [40]:
retriever = test_engine.get_retriever("scene1Classroom_part2_thanks_similarity")

retriever.search("Hola", 3)


(array([42, 27, 14], dtype=int32),
 array([0.99999994, 0.5333565 , 0.5254723 ], dtype=float32),
 array(['Hola', 'Buenas! Soy [UNK] encantado.', 'Holaaa, soy [UNK] que ta'],
       dtype=object))